## **Laboratorio: Uso de la API de un LLM con OpenAI**

En este laboratorio trabajarás directamente con la API de OpenAI para entender **cómo se comunica un programa con un LLM** y qué palancas tienes para controlar su comportamiento. Esto conecta con los conceptos vistos en clase: qué recibe realmente el modelo, cómo los parámetros de generación afectan la salida, cómo redactar prompts efectivos, por qué el modelo puede fallar y cómo evaluar sus respuestas de forma sistemática.

## ¿Qué practicarás?

- **Llamadas a la API** — enviar texto al modelo y recibir una respuesta estructurada.
- **Parámetros de generación** — `max_output_tokens`, `temperature` y `top_p`: qué hacen y cómo se nota su efecto.
- **Streaming** — recibir la respuesta token a token, como en ChatGPT.
- **Prompt Engineering** — la diferencia entre un prompt vago y uno bien construido.
- **Salida estructurada en JSON** — extraer información de texto no estructurado.
- **Fallos del modelo** — alucinaciones y respuestas sin contexto suficiente.
- **Evaluación básica** — medir la calidad de las respuestas de forma manual y automática (LLM-as-judge).

## Objetivos del Laboratorio

Al finalizar este laboratorio serás capaz de:

1. **Conectarte a la API de OpenAI** usando la librería oficial de Python.
2. **Controlar los parámetros de generación** y entender su impacto en la salida.
3. **Implementar streaming** para mejorar la experiencia de usuario en aplicaciones reales.
4. **Diseñar prompts efectivos** aplicando las dimensiones vistas en clase (rol, contexto, objetivo, formato, restricciones).
5. **Extraer datos estructurados** de texto libre usando JSON.
6. **Identificar y mitigar fallos** del modelo (alucinaciones, falta de contexto).
7. **Evaluar respuestas** manualmente y usando el propio modelo como juez.

**Duración estimada:** 2-3 horas

---

## Sección 1: Configuración

Antes de poder llamar a la API necesitamos instalar la librería oficial de OpenAI y configurar nuestra clave de acceso. La clave identifica tu cuenta y se usa para facturar el consumo de tokens.

**Celda 1: Instalación de la librería OpenAI**

In [ ]:
!pip install openai

La librería `openai` es el cliente oficial de Python para la API de OpenAI. Incluye soporte para la Responses API, streaming, manejo de errores y reintentos automáticos. Si ya la tienes instalada, este comando simplemente verificará que está actualizada.

**Celda 2: Configuración de la API Key y creación del cliente**

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "sk-..."  # ← Reemplaza con tu clave real

client = OpenAI(api_key=OPENAI_API_KEY)

# Verificación: listar los primeros modelos disponibles
modelos = list(client.models.list())
print(f"Conexión OK — {len(modelos)} modelos disponibles en tu cuenta.")
print("Ejemplo:", modelos[0].id)

El objeto `OpenAI(api_key=...)` es el punto de entrada a la API. Todas las llamadas al modelo se realizan a través de este cliente. Si la clave es incorrecta, `client.models.list()` lanzará un error `AuthenticationError` en este punto, antes de consumir tokens.

**Celda 3: Primera llamada — respuesta simple e información de uso**

In [ ]:
response = client.responses.create(
    model="gpt-4o-mini",
    input="Explica qué es un LLM en 3 bullets."
)

print("=== RESPUESTA ===")
print(response.output_text)

print("\n=== USO DE TOKENS ===")
print(f"  Tokens de entrada (prompt):  {response.usage.input_tokens}")
print(f"  Tokens de salida (respuesta): {response.usage.output_tokens}")
print(f"  Total:                        {response.usage.total_tokens}")

`client.responses.create()` es el método principal de la Responses API. El parámetro `input` acepta una cadena de texto o una lista de mensajes. El objeto devuelto contiene `output_text` (la respuesta como string) y `usage` (cuántos tokens se consumieron). Los tokens son la unidad de facturación: aproximadamente 1 token ≈ 0,75 palabras en inglés, algo menos en español.

---
## Sección 2: Parámetros de Generación

El modelo tiene varios parámetros que controlan **cómo genera el texto**. En clase vimos que la generación es un proceso de muestreo sobre una distribución de probabilidad de tokens. Los parámetros de esta sección modifican ese proceso. Dominarlos es esencial para obtener respuestas útiles y reproducibles.

**Celda 4: max_output_tokens — controlar la longitud de la respuesta**

In [ ]:
prompt = "Explica qué es un Transformer."

# Respuesta corta: máximo 30 tokens de salida
resp_corta = client.responses.create(
    model="gpt-4o-mini",
    input=prompt,
    max_output_tokens=30
)

# Respuesta larga: máximo 200 tokens de salida
resp_larga = client.responses.create(
    model="gpt-4o-mini",
    input=prompt,
    max_output_tokens=200
)

print("=== max_output_tokens=30 ===")
print(resp_corta.output_text)
print(f"  [Tokens de salida usados: {resp_corta.usage.output_tokens}]")

print("\n=== max_output_tokens=200 ===")
print(resp_larga.output_text)
print(f"  [Tokens de salida usados: {resp_larga.usage.output_tokens}]")

`max_output_tokens` establece un límite **duro** en la longitud de la respuesta. Si el modelo llega al límite antes de terminar la idea, la respuesta quedará truncada. Esto tiene dos implicaciones prácticas: (1) controla el coste máximo por llamada, y (2) puede degradar la calidad si el límite es demasiado bajo para la tarea. Observa en la respuesta corta cómo la frase puede quedar incompleta.

**Preguntas de reflexión:**
- ¿La respuesta con 30 tokens quedó completa o truncada?
- ¿Cuántos tokens usa realmente la respuesta larga? ¿Llega al límite de 200?
- ¿En qué tipo de aplicación usarías un `max_output_tokens` bajo? ¿Y uno alto?

**Celda 5: temperature — creatividad vs. determinismo**

In [ ]:
prompt = "Dame 5 nombres creativos para una app de recetas saludables."

print("=" * 60)
print("temperature=0.1 — Ejecución 1")
print("=" * 60)
r = client.responses.create(model="gpt-4o-mini", input=prompt, temperature=0.1)
print(r.output_text)

print("\n" + "=" * 60)
print("temperature=0.1 — Ejecución 2 (¿igual que la anterior?)")
print("=" * 60)
r = client.responses.create(model="gpt-4o-mini", input=prompt, temperature=0.1)
print(r.output_text)

print("\n" + "=" * 60)
print("temperature=1.3 — Ejecución 1")
print("=" * 60)
r = client.responses.create(model="gpt-4o-mini", input=prompt, temperature=1.3)
print(r.output_text)

print("\n" + "=" * 60)
print("temperature=1.3 — Ejecución 2 (¿diferente a la anterior?)")
print("=" * 60)
r = client.responses.create(model="gpt-4o-mini", input=prompt, temperature=1.3)
print(r.output_text)

`temperature` controla **cuánto aleatoriedad** se introduce en el muestreo de tokens. Con `temperature=0.0` el modelo siempre elige el token más probable (respuesta casi determinista). Con `temperature=1.3` los tokens menos probables tienen mucho más peso, generando respuestas más variadas y creativas, pero también menos coherentes. El rango habitual de uso es 0.0–1.0; valores por encima de 1.0 son más experimentales.

**Preguntas de reflexión:**
- ¿Son las dos ejecuciones con `temperature=0.1` idénticas o muy similares?
- ¿Son las dos ejecuciones con `temperature=1.3` claramente diferentes?
- ¿Para qué tareas usarías temperatura baja? ¿Para cuáles alta?

**Celda 6: top_p — muestreo por núcleo**

In [ ]:
prompt = "Dame 5 nombres creativos para una app de recetas saludables."

# top_p=1.0: considera el 100% de los tokens posibles (comportamiento por defecto)
print("=== top_p=1.0 (vocabulario completo) ===")
r = client.responses.create(model="gpt-4o-mini", input=prompt, top_p=1.0)
print(r.output_text)

# top_p=0.2: considera solo los tokens cuya probabilidad acumulada supera el 20%
# (es decir, solo los tokens más probables)
print("\n=== top_p=0.2 (solo tokens más probables) ===")
r = client.responses.create(model="gpt-4o-mini", input=prompt, top_p=0.2)
print(r.output_text)

`top_p` (nucleus sampling) limita el conjunto de tokens candidatos a aquellos cuya **probabilidad acumulada** alcanza el umbral `p`. Con `top_p=0.2` solo los tokens más probables participan en el muestreo, haciendo la respuesta más conservadora y repetible. Con `top_p=1.0` todos los tokens son candidatos. OpenAI recomienda ajustar `temperature` o `top_p`, pero no los dos a la vez, ya que sus efectos se componen de forma no intuitiva.

**Preguntas de reflexión:**
- ¿Notas diferencia en el estilo o vocabulario entre `top_p=1.0` y `top_p=0.2`?
- ¿En qué se diferencia conceptualmente `top_p` de `temperature`?

---
## Sección 3: Streaming

Por defecto, la API espera a que el modelo genere la respuesta completa antes de devolvértela. Con **streaming**, recibes los tokens según se van generando, igual que en la interfaz de ChatGPT. Esto mejora la experiencia de usuario en aplicaciones interactivas sin aumentar el coste.

**Celda 7: Respuesta en streaming token a token**

In [ ]:
prompt = "Explica en 8 pasos cómo funciona un LLM, del prompt a la respuesta."

print("Respuesta en streaming:\n")

with client.responses.stream(model="gpt-4o-mini", input=prompt) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n[Streaming completado]")

`client.responses.stream()` devuelve un gestor de contexto. Dentro del bloque `with`, iteramos sobre `stream.text_stream`, que genera fragmentos de texto (chunks) conforme el modelo los produce. `flush=True` fuerza que cada fragmento se imprima inmediatamente sin esperar a llenar el buffer del terminal. El coste en tokens es exactamente el mismo que sin streaming: la diferencia es solo cuándo recibes la respuesta.

---
## Sección 4: Prompt Engineering

En clase vimos que un buen prompt incluye: **rol** (quién es el modelo), **contexto** (situación relevante), **objetivo** (qué debe hacer), **formato** (cómo debe presentar la respuesta), **restricciones** (qué debe evitar) y opcionalmente **ejemplos**. En esta sección verás el impacto real de aplicar estas dimensiones.

**Celda 8: Comparación — prompt malo vs. prompt bueno**

In [ ]:
# Prompt vago: sin rol, sin formato, sin restricciones
prompt_malo = "Háblame de IA"

# Prompt estructurado: rol + contexto + objetivo + formato + restricciones
prompt_bueno = """Actúa como un profesor universitario de Inteligencia Artificial.
Necesito explicar a alumnos de máster qué es la IA y sus principales ramas.
Contexto: los alumnos tienen conocimientos de programación pero poca experiencia en IA.
Devuelve la respuesta en formato de lista con 5 puntos, cada uno con título en negrita y
una explicación de 2 frases.
Ten en cuenta: no uses jerga excesivamente técnica, céntrate en aplicaciones prácticas."""

resp_malo  = client.responses.create(model="gpt-4o-mini", input=prompt_malo)
resp_bueno = client.responses.create(model="gpt-4o-mini", input=prompt_bueno)

print("=" * 60)
print("PROMPT MALO (vago):")
print("=" * 60)
print(resp_malo.output_text)

print("\n" + "=" * 60)
print("PROMPT BUENO (estructurado):")
print("=" * 60)
print(resp_bueno.output_text)

Observa las diferencias en longitud, estructura y relevancia. El prompt malo produce una respuesta genérica porque el modelo tiene que adivinar qué audiencia, qué nivel de detalle y qué formato se espera. El prompt bueno especifica todos esos parámetros explícitamente, lo que reduce la ambigüedad y mejora la utilidad de la respuesta sin aumentar el coste computacional.

**Celda 9: Tu turno — diseña tu propio prompt estructurado**

In [ ]:
# Diseña un prompt para una tarea de tu elección
# Rellena cada componente con contenido real (no dejes los placeholders)

prompt_alumno = """
Actúa como un experto en marketing digital con 10 años de experiencia.
Necesito una estrategia de contenido para redes sociales para una startup de tecnología educativa.
Contexto: la startup acaba de lanzar una app de aprendizaje de idiomas con IA, con presupuesto limitado.
Devuelve la respuesta como un plan semanal con 5 tipos de publicaciones, indicando plataforma y objetivo de cada una.
Ten en cuenta: el tono debe ser cercano y motivador, evita el lenguaje corporativo formal.
"""

respuesta = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_alumno
)

print(respuesta.output_text)

El template de prompt que acabas de rellenar aplica las seis dimensiones del framework visto en clase. Compara la calidad de la respuesta con la del prompt malo de la celda anterior. En aplicaciones reales, los prompts suelen evolucionar iterativamente: se prueban, se refinan y se versionan igual que el código.

---
## Sección 5: Salida Estructurada en JSON

Cuando queremos usar la salida del modelo en código (guardarla en una base de datos, pasarla a otro sistema, mostrarla en una interfaz), necesitamos que la respuesta tenga un formato predecible. Pedir JSON en el prompt es el método más directo para conseguirlo.

**Celda 10: Extracción de información estructurada de una reseña**

In [ ]:
import json

resena = """
Compré el portátil UltraBook Pro el 15 de marzo y la verdad es que estoy bastante
decepcionado. La batería dura mucho menos de lo prometido y el teclado empieza a
fallar después de solo dos semanas de uso. No lo recomendaría para uso profesional.
"""

prompt = f"""Extrae la siguiente información de la reseña y devuelve ÚNICAMENTE un objeto JSON
válido, sin texto adicional antes ni después:

{{
  "cliente": "nombre o 'anónimo' si no se menciona",
  "producto": "nombre del producto",
  "fecha": "fecha de compra en formato YYYY-MM-DD o null si no se menciona",
  "sentimiento": "positivo | negativo | neutro"
}}

Reseña:
{resena}"""

respuesta = client.responses.create(
    model="gpt-4o-mini",
    input=prompt,
    temperature=0.0  # máximo determinismo para extracción de datos
)

# Parsear la respuesta como JSON
datos = json.loads(respuesta.output_text)

print("=== Datos extraídos ===")
print(f"  Cliente:     {datos['cliente']}")
print(f"  Producto:    {datos['producto']}")
print(f"  Fecha:       {datos['fecha']}")
print(f"  Sentimiento: {datos['sentimiento']}")
print("\nJSON completo:")
print(json.dumps(datos, ensure_ascii=False, indent=2))

Al usar `temperature=0.0` minimizamos la variabilidad en la extracción de datos, donde queremos precisión, no creatividad. La instrucción `"devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional"` es importante porque los modelos tienden a añadir frases introductorias ("Aquí tienes el JSON:") que romperían `json.loads()`. Para aplicaciones en producción, considera la opción `response_format={"type": "json_object"}` que fuerza la salida en JSON a nivel de API.

**Celda 11: Cambia el texto y añade un nuevo campo de extracción**

In [ ]:
# Reseña diferente para probar la extracción
resena_nueva = """
Llevaba meses buscando unos auriculares inalámbricos y finalmente compré los SoundMax Elite
en enero. ¡Increíbles! La calidad de sonido es espectacular y la cancelación de ruido funciona
de maravilla. Los uso 8 horas al día en el trabajo y la batería aguanta perfectamente.
Totalmente recomendados. — Carlos G.
"""

# Prompt actualizado con un campo adicional: 'recomendaria'
prompt_nuevo = f"""Extrae la siguiente información de la reseña y devuelve ÚNICAMENTE un objeto JSON
válido, sin texto adicional antes ni después:

{{
  "cliente": "nombre o 'anónimo' si no se menciona",
  "producto": "nombre del producto",
  "fecha": "fecha de compra en formato YYYY-MM-DD o null si no se menciona",
  "sentimiento": "positivo | negativo | neutro",
  "recomendaria": true o false según si el cliente recomienda el producto
}}

Reseña:
{resena_nueva}"""

respuesta_nueva = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_nuevo,
    temperature=0.0
)

datos_nuevos = json.loads(respuesta_nueva.output_text)

print("=== Datos extraídos de la nueva reseña ===")
for campo, valor in datos_nuevos.items():
    print(f"  {campo}: {valor}")

Añadir un nuevo campo al esquema JSON es tan sencillo como añadirlo al prompt. El modelo infiere el valor correcto para `recomendaria` a partir del texto, sin que haya una mención explícita a la palabra "recomendar" en todos los casos. Esta capacidad de inferencia es una de las diferencias clave entre los LLMs y los sistemas de extracción basados en reglas.

---
## Sección 6: El LLM puede fallar

En clase vimos que los LLMs son modelos de lenguaje, no bases de datos. No tienen acceso a información privada de tu empresa, ni a eventos posteriores a su fecha de entrenamiento. Cuando se les pregunta sobre algo que no saben, pueden **alucinar** (inventar una respuesta plausible pero falsa) o reconocer honestamente su límite. Esta sección muestra ambos escenarios.

**Celda 12: Pregunta sin contexto — el modelo no tiene la información**

In [ ]:
pregunta_sin_contexto = "¿Cuál es la política de devoluciones de nuestra empresa?"

respuesta_sin_contexto = client.responses.create(
    model="gpt-4o-mini",
    input=pregunta_sin_contexto
)

print("=== RESPUESTA SIN CONTEXTO ===")
print(respuesta_sin_contexto.output_text)

El modelo no sabe nada de "nuestra empresa" porque esa información no existe en sus datos de entrenamiento. Dependiendo del modelo y del prompt, puede responder reconociendo su ignorancia, o puede inventar una política de devoluciones genérica que suene plausible pero sea incorrecta. Este es el problema de las **alucinaciones**: el modelo genera texto coherente aunque carezca de fundamento factual.

**Celda 13: La misma pregunta con contexto — respuesta fundamentada**

In [ ]:
# Política ficticia de devoluciones que inyectamos como contexto en el prompt
contexto_empresa = """
POLÍTICA DE DEVOLUCIONES — TechStore S.L. (vigente desde enero 2024):
- Los clientes tienen 30 días naturales desde la fecha de entrega para solicitar una devolución.
- El producto debe estar en su embalaje original y sin signos de uso.
- Las devoluciones se gestionan a través del portal web o llamando al 900 123 456.
- El reembolso se realiza en el mismo método de pago original en un plazo de 5-7 días hábiles.
- Los productos personalizados y el software descargado no son elegibles para devolución.
"""

prompt_con_contexto = f"""Usa ÚNICAMENTE la siguiente información para responder la pregunta.
Si la respuesta no está en el contexto, indícalo explícitamente.

CONTEXTO:
{contexto_empresa}

PREGUNTA: ¿Cuál es la política de devoluciones de nuestra empresa?"""

respuesta_con_contexto = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_con_contexto
)

print("=== RESPUESTA CON CONTEXTO ===")
print(respuesta_con_contexto.output_text)

Este patrón se llama **RAG (Retrieval-Augmented Generation)**: en lugar de confiar en lo que el modelo memorizó durante el entrenamiento, le proporcionamos la información relevante directamente en el prompt. La instrucción `"Usa ÚNICAMENTE la siguiente información"` ancla la respuesta al contexto proporcionado y reduce drásticamente las alucinaciones. Es la técnica más usada en aplicaciones empresariales de LLMs.

**Preguntas de reflexión:**
- ¿Qué respondió el modelo en la Celda 12? ¿Reconoció su ignorancia o alucinó?
- ¿Cómo cambió la respuesta al añadir el contexto?
- ¿Qué pasaría si el usuario pregunta por algo que no está en el contexto?

---
## Sección 7: Evaluación Básica

En clase vimos que evaluar un LLM no es trivial: no hay una respuesta correcta única para la mayoría de tareas. En esta sección practicamos dos enfoques: evaluación **manual** (el humano puntúa) y evaluación **automática** usando el propio modelo como juez (LLM-as-judge).

**Celda 14: Definición de casos de prueba y generación de respuestas**

In [ ]:
# Casos de prueba: pregunta + criterio de calidad esperado
casos = [
    {
        "pregunta": "Explica la diferencia entre machine learning y deep learning en 2 frases.",
        "criterio": "La respuesta debe mencionar que deep learning es un subconjunto de ML "
                    "y hacer referencia a redes neuronales. Debe tener exactamente 2 frases."
    },
    {
        "pregunta": "¿Cuáles son los 3 tipos principales de aprendizaje automático?",
        "criterio": "La respuesta debe mencionar supervisado, no supervisado y por refuerzo. "
                    "Debe dar una breve descripción de cada uno."
    },
    {
        "pregunta": "Escribe un ejemplo de uso de IA en el sector sanitario en formato JSON con "
                    "campos: aplicacion, beneficio, riesgo.",
        "criterio": "La respuesta debe ser un JSON válido con exactamente los tres campos indicados. "
                    "La aplicación debe ser realista y relevante en sanidad."
    }
]

# Generar una respuesta del modelo para cada caso
respuestas = []
for i, caso in enumerate(casos):
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=caso["pregunta"]
    )
    respuestas.append(resp.output_text)
    print(f"\n{'='*60}")
    print(f"CASO {i} — {caso['pregunta'][:60]}...")
    print(f"{'='*60}")
    print(resp.output_text)

Cada caso de prueba tiene una `pregunta` y un `criterio` que define qué hace que una respuesta sea buena. Este enfoque, llamado **test suite**, permite evaluar el modelo de forma sistemática y reproducible. Con el mismo conjunto de casos podemos comparar diferentes modelos, diferentes prompts o diferentes versiones de la API.

**Celda 15: Evaluación manual — puntúa las respuestas**

In [ ]:
# Evalúa cada respuesta según tu criterio (1=malo, 2=regular, 3=aceptable, 4=bueno, 5=excelente)
# Puntuaciones asignadas tras revisar las respuestas de la Celda 14

evaluaciones = [
    {"caso": 0, "calidad": 4, "sigue_instrucciones": 5, "formato_correcto": 5},
    {"caso": 1, "calidad": 5, "sigue_instrucciones": 5, "formato_correcto": 4},
    {"caso": 2, "calidad": 4, "sigue_instrucciones": 4, "formato_correcto": 5},
]

print("=== Evaluación Manual ===")
print(f"{'Caso':<6} {'Calidad':<10} {'Instrucciones':<16} {'Formato':<10} {'Media':<8}")
print("-" * 55)

puntuaciones_globales = []
for ev in evaluaciones:
    media = (ev["calidad"] + ev["sigue_instrucciones"] + ev["formato_correcto"]) / 3
    puntuaciones_globales.append(media)
    print(f"{ev['caso']:<6} {ev['calidad']:<10} {ev['sigue_instrucciones']:<16} {ev['formato_correcto']:<10} {media:.2f}")

media_global = sum(puntuaciones_globales) / len(puntuaciones_globales)
print("-" * 55)
print(f"Media global del modelo: {media_global:.2f} / 5.00")

La evaluación manual es el método más fiable pero más costoso: requiere que un humano lea cada respuesta y aplique un criterio. Para proyectos pequeños o durante el desarrollo inicial es suficiente. Para escalar a cientos o miles de casos necesitamos automatización, que es lo que haremos en la siguiente celda.

**Celda 16: LLM-as-judge — el modelo evalúa sus propias respuestas**

In [ ]:
print("=== Evaluación Automática (LLM-as-judge) ===\n")

for i, (caso, respuesta_modelo) in enumerate(zip(casos, respuestas)):
    prompt_juez = f"""Eres un evaluador experto de sistemas de IA. Tu tarea es puntuar la respuesta
de un modelo de lenguaje según el criterio proporcionado.

PREGUNTA ORIGINAL:
{caso['pregunta']}

CRITERIO DE EVALUACIÓN:
{caso['criterio']}

RESPUESTA DEL MODELO:
{respuesta_modelo}

Evalúa la respuesta con una puntuación del 1 al 5 donde:
1 = No cumple el criterio en absoluto
3 = Cumple el criterio parcialmente
5 = Cumple el criterio completamente

Devuelve ÚNICAMENTE un JSON con este formato exacto:
{{"puntuacion": X, "justificacion": "Una frase explicando la puntuación"}}"""

    resp_juez = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_juez,
        temperature=0.0
    )

    evaluacion = json.loads(resp_juez.output_text)

    print(f"Caso {i}: {caso['pregunta'][:55]}...")
    print(f"  Puntuación LLM-juez: {evaluacion['puntuacion']} / 5")
    print(f"  Justificación: {evaluacion['justificacion']}")
    print()

**LLM-as-judge** es una técnica cada vez más usada en la industria para evaluar respuestas de LLMs a escala. En lugar de pagar a anotadores humanos para revisar miles de respuestas, se usa otro modelo (o el mismo) como evaluador automático. Sus limitaciones incluyen: posible sesgo hacia respuestas verbosas, dificultad para detectar errores factuales sutiles y el llamado "sesgo de posición" (tiende a favorecer la primera opción en comparaciones). A pesar de estas limitaciones, correlaciona bien con la evaluación humana para muchas tareas.

---
## Conclusiones

En este laboratorio has trabajado directamente con la API de OpenAI y has experimentado los conceptos vistos en clase. Aquí tienes un resumen de todo lo practicado:

| Concepto | Qué aprendiste | Celda |
|----------|---------------|-------|
| **API y cliente** | Cómo conectarse y hacer la primera llamada | 2, 3 |
| **Tokens y coste** | Qué son, cómo contarlos, cómo controlarlos | 3, 4 |
| **max_output_tokens** | Límite de longitud de respuesta | 4 |
| **temperature** | Creatividad vs. determinismo en la generación | 5 |
| **top_p** | Nucleus sampling para controlar el vocabulario | 6 |
| **Streaming** | Recibir tokens en tiempo real | 7 |
| **Prompt Engineering** | Las 6 dimensiones de un prompt efectivo | 8, 9 |
| **Salida JSON** | Extracción de datos estructurados | 10, 11 |
| **Alucinaciones** | Cómo fallan los modelos sin contexto | 12 |
| **RAG básico** | Mitigar fallos inyectando contexto en el prompt | 13 |
| **Evaluación manual** | Métricas humanas con criterios explícitos | 14, 15 |
| **LLM-as-judge** | Evaluación automática a escala | 16 |

### Próximos pasos

- Explora la **Assistants API** de OpenAI para conversaciones multi-turno con memoria.
- Investiga **function calling** para que el modelo decida cuándo llamar a funciones externas.
- Profundiza en **RAG** conectando el modelo a una base de datos vectorial (Chroma, Pinecone, pgvector).
- Consulta el [Playground de OpenAI](https://platform.openai.com/playground) para experimentar con prompts de forma interactiva.

---
**Laboratorio: Uso de la API de un LLM con OpenAI — Máster en IA / EBIS**